**Использование генеративного ИИ**: В рамках заданий использование генеративного ИИ регулируется теми же правилами, что и совместная работа. Как и при взаимодействии с другими участниками, каждый студент должен самостоятельно записать решения, не опираясь напрямую на результат взаимодействия, а в работе следует указать характер такого сотрудничества. Использование инструментов генеративного ИИ для существенного выполнения частей заданий противоречит их замыслу и является нарушением [Кодекса чести](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).

In [ ]:
# Эта ячейка нужна, если вы используете Colab
# Подключает Google Drive к виртуальной машине Colab.
from google.colab import drive
drive.mount('/content/drive')

# TODO: укажите имя папки на Google Drive, в которой сохранена распакованная
# папка задания, например 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# После подключения Google Drive этот код позволяет интерпретатору Python
# виртуальной машины Colab загружать из неё Python-файлы.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# Загружает набор данных CIFAR-10 на Google Drive,
# если он ещё не существует.
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Упражнение по классификатору Softmax

В этом упражнении вы:

- реализуете полностью векторизованную **функцию потерь** классификатора Softmax;
- реализуете полностью векторизованное выражение для её **аналитического градиента**;
- **проверите реализацию** с помощью численного градиента;
- подберёте **скорость обучения и силу регуляризации** с использованием проверочной выборки;
- **оптимизируете** функцию потерь с помощью **SGD**;
- **визуализируете** итоговые обученные веса.

In [ ]:
# Выполняем начальную настройку для этого ноутбука.
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# Это небольшая магия, благодаря которой графики matplotlib отображаются прямо
# в ноутбуке, а не в отдельном окне.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # задаём размер графиков по умолчанию
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Ещё немного магии: ноутбук будет автоматически перезагружать внешние Python-модули;
# см. http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## Загрузка и предобработка данных CIFAR-10

In [ ]:
# Загружаем исходные данные CIFAR-10.
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# Очищаем переменные, чтобы не загружать данные несколько раз и не вызвать проблему с памятью.
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# Для проверки выводим размеры обучающих и тестовых данных.
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# Визуализируем несколько примеров из набора данных.
# Показываем несколько обучающих изображений из каждого класса.
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Разделяем данные на обучающую, проверочную и тестовую выборки. Также создаём
# небольшую отладочную выборку как подмножество обучающих данных, чтобы код
# быстрее выполнялся во время разработки.
num_training = 49000
num_validation = 1000
num_test = 1000
num_dev = 500

# Проверочная выборка состоит из num_validation точек исходной обучающей выборки.
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Обучающая выборка состоит из первых num_training точек исходной обучающей выборки.
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# Также создаём отладочную выборку --- небольшое подмножество обучающей выборки.
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# Используем первые num_test точек исходной тестовой выборки как тестовую выборку.
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# Предобработка: преобразуем данные изображений в строки.
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# Для проверки выводим размерности данных.
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)

In [ ]:
# Предобработка: вычитаем среднее изображение.
# Сначала вычисляем среднее изображение по обучающим данным.
mean_image = np.mean(X_train, axis=0)
print(mean_image[:10]) # выводим несколько элементов
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # визуализируем среднее изображение
plt.show()

# Затем вычитаем среднее изображение из обучающих и тестовых данных.
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# Наконец, добавляем размерность из единиц для смещения (bias trick), чтобы
# классификатору нужно было оптимизировать только одну матрицу весов W.
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])

print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)

## Классификатор Softmax

Код этого раздела необходимо написать в `cs231n/classifiers/softmax.py`.

Как видно, функция `softmax_loss_naive`, использующая циклы для вычисления функции потерь Softmax, уже заполнена.

In [ ]:
# Вычисляем функцию потерь с помощью предоставленной наивной реализации:
from cs231n.classifiers.softmax import softmax_loss_naive
import time

# Создаём случайную матрицу малых весов классификатора Softmax.
W = np.random.randn(3073, 10) * 0.0001

loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
print('loss: %f' % (loss, ))

# Для грубой проверки значение функции потерь должно быть близко к -log(0.1).
print('loss: %f' % loss)
print('sanity check: %f' % (-np.log(0.1)))

**Контрольный вопрос 1**

Почему ожидается, что функция потерь будет близка к $-\log(0.1)$? Кратко объясните.

$\color{blue}{\textit Ваш ответ:}$ *заполните здесь*

Возвращаемый функцией выше `grad` сейчас полностью состоит из нулей. Выведите и реализуйте градиент функции потерь Softmax непосредственно в функции `softmax_loss_naive`. Будет удобно добавить новый код внутрь уже существующей функции.

Чтобы убедиться, что градиент реализован правильно, можно численно оценить градиент функции потерь и сравнить численную оценку с вычисленным градиентом. Для этого уже предоставлен следующий код:

In [ ]:
# После реализации градиента пересчитайте его кодом ниже
# и проверьте градиент предоставленной функцией.

# Вычисляем функцию потерь и её градиент при W.
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# Численно вычисляем градиент для нескольких случайно выбранных размерностей и
# сравниваем его с аналитически вычисленным градиентом. Числа должны почти точно
# совпадать по всем размерностям.
from cs231n.gradient_check import grad_check_sparse
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad)

# Повторяем проверку градиента при включённой регуляризации.
# Не забудьте добавить градиент регуляризации.
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad)

**Контрольный вопрос 2**

Хотя проверка градиента надёжна для функции потерь Softmax, для функции потерь SVM иногда одна из размерностей при проверке может не совпасть точно. Чем может быть вызвано такое расхождение? Есть ли повод для беспокойства? Приведите простой одномерный пример, в котором проверка градиента SVM может не пройти. Как изменение величины зазора повлияет на частоту такого явления?

Обратите внимание, что функция потерь SVM для примера $(x_i, y_i)$ определена так: $$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta),$$ где $j$ перебирает все классы, кроме правильного класса $y_i$, а $s_j$ обозначает оценку классификатора для $j$-го класса. $\Delta$ --- скалярный зазор. Подробнее см. раздел «Multiclass Support Vector Machine loss» на [этой странице](https://cs231n.github.io/linear-classify/).

*Подсказка: строго говоря, функция потерь SVM не является дифференцируемой.*


$\color{blue}{\textit Ваш ответ:}$ *заполните здесь.*

In [ ]:
# Затем реализуйте функцию softmax_loss_vectorized; пока вычисляйте только функцию
# потерь, градиент будет реализован далее.
tic = time.time()
loss_naive, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from cs231n.classifiers.softmax import softmax_loss_vectorized
tic = time.time()
loss_vectorized, _ = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# Функции потерь должны совпадать, но векторизованная реализация должна быть намного быстрее.
print('difference: %f' % (loss_naive - loss_vectorized))

In [ ]:
# Завершите реализацию softmax_loss_vectorized и вычислите градиент
# функции потерь векторизованным способом.

# Наивная и векторизованная реализации должны давать одинаковый результат,
# но векторизованная версия должна оставаться значительно быстрее.
tic = time.time()
_, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss and gradient: computed in %fs' % (toc - tic))

tic = time.time()
_, grad_vectorized = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss and gradient: computed in %fs' % (toc - tic))

# Функция потерь --- одно число, поэтому значения, вычисленные двумя
# реализациями, сравнить легко. Градиент, напротив, является матрицей,
# поэтому для сравнения используем норму Фробениуса.
difference = np.linalg.norm(grad_naive - grad_vectorized, ord='fro')
print('difference: %f' % difference)

### Стохастический градиентный спуск

Теперь у нас есть векторизованные и эффективные выражения для функции потерь и градиента, а вычисленный градиент совпадает с численным. Значит, можно применить SGD для минимизации функции потерь. Код этой части необходимо написать в `cs231n/classifiers/linear_classifier.py`.

In [ ]:
# В файле linear_classifier.py реализуйте SGD в функции
# LinearClassifier.train(), затем запустите её с кодом ниже.
from cs231n.classifiers import Softmax
softmax = Softmax()
tic = time.time()
loss_hist = softmax.train(X_train, y_train, learning_rate=1e-7, reg=2.5e4,
                      num_iters=1500, verbose=True)
toc = time.time()
print('That took %fs' % (toc - tic))

In [ ]:
# Для отладки полезно построить график функции потерь в зависимости от
# номера итерации:
plt.plot(loss_hist)
plt.xlabel('Номер итерации')
plt.ylabel('Значение функции потерь')
plt.show()

In [ ]:
# Напишите функцию LinearClassifier.predict и оцените качество
# на обучающей и проверочной выборках.
# Точность на проверочной выборке должна быть около 0.34 (> 0.33).
y_train_pred = softmax.predict(X_train)
print('training accuracy: %f' % (np.mean(y_train == y_train_pred), ))
y_val_pred = softmax.predict(X_val)
print('validation accuracy: %f' % (np.mean(y_val == y_val_pred), ))

In [ ]:
# Сохраняем обученную модель 
softmax.save("softmax.npy")

In [ ]:
# Используйте проверочную выборку для подбора гиперпараметров: силы регуляризации
# и скорости обучения. Попробуйте разные диапазоны значений; при аккуратном подборе
# точность классификации на проверочной выборке должна быть около 0.365 (> 0.36).

# Примечание: при подборе гиперпараметров могут появиться предупреждения о времени
# выполнения или переполнении. Их причиной могут быть экстремальные значения, это не ошибка.

# results --- словарь, сопоставляющий кортежи вида
# (learning_rate, regularization_strength) к кортежам вида
# (training_accuracy, validation_accuracy). Точность равна доле
# правильно классифицированных точек данных.
results = {}
best_val = -1   # Наивысшая точность на проверочной выборке, найденная к этому моменту.
best_softmax = None # Объект Softmax с наивысшей точностью на проверочной выборке.

################################################################################
# TODO:                                                                        #
# Напишите код, который выбирает лучшие гиперпараметры по проверочной выборке. #
# Для каждой комбинации гиперпараметров обучите Softmax на обучающей выборке,  #
# вычислите точность на обучающей и проверочной выборках и сохраните эти       #
# значения в словаре results. Кроме того, сохраните лучшую точность на        #
# проверочной выборке в best_val и объект Softmax, который её достигает, в    #
# best_softmax.                                                               #
#                                                                              #
# Подсказка: во время разработки кода проверки используйте небольшое значение #
# num_iters, чтобы обучение классификаторов занимало мало времени. Убедившись  #
# в работе кода, перезапустите его с большим значением num_iters.              #
################################################################################

# Приведено для справки. Эти гиперпараметры можно изменить.
learning_rates = [1e-7, 1e-6]
regularization_strengths = [2.5e4, 1e4]



# Выводим результаты.
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during cross-validation: %f' % best_val)

In [ ]:
# Визуализируем результаты кросс-валидации.
import math
import pdb

# pdb.set_trace()

x_scatter = [math.log10(x[0]) for x in results]
y_scatter = [math.log10(x[1]) for x in results]

# Строим график точности на обучающей выборке.
marker_size = 100
colors = [results[x][0] for x in results]
plt.subplot(2, 1, 1)
plt.tight_layout(pad=3)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('Логарифм скорости обучения')
plt.ylabel('Логарифм силы регуляризации')
plt.title('Точность CIFAR-10 на обучающей выборке')

# Строим график точности на проверочной выборке.
colors = [results[x][1] for x in results] # размер маркеров по умолчанию равен 20
plt.subplot(2, 1, 2)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('Логарифм скорости обучения')
plt.ylabel('Логарифм силы регуляризации')
plt.title('Точность CIFAR-10 на проверочной выборке')
plt.show()

In [ ]:
# Оцениваем лучшую модель Softmax на тестовой выборке.
y_test_pred = best_softmax.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('Softmax classifier on raw pixels final test set accuracy: %f' % test_accuracy)

In [ ]:
# Сохраняем лучшую модель Softmax.
best_softmax.save("best_softmax.npy")

In [ ]:
# Визуализируем обученные веса для каждого класса.
# В зависимости от выбранных скорости обучения и силы регуляризации они могут
# выглядеть более или менее наглядно.
w = best_softmax.W[:-1,:] # исключаем компоненту смещения
w = w.reshape(32, 32, 3, 10)
w_min, w_max = np.min(w), np.max(w)
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # Масштабируем веса к диапазону от 0 до 255.
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])

**Контрольный вопрос 3**

Опишите, как выглядят визуализированные веса классификатора Softmax, и кратко объясните, почему они имеют такой вид.

$\color{blue}{\textit Ваш ответ:}$ *заполните здесь*

**Контрольный вопрос 4** --- *Верно или неверно*

Предположим, что общая функция потерь при обучении определена как сумма потерь по отдельным точкам данных для всех обучающих примеров. Можно добавить новую точку данных в обучающую выборку так, что функция потерь Softmax изменится, а функция потерь SVM останется неизменной.

$\color{blue}{\textit Ваш ответ:}$


$\color{blue}{\textit Ваше объяснение:}$